# Task 4 — The Information Bottleneck Problem

## 1. Squeezing Meaning

In a basic Seq2Seq model, the Encoder compresses the entire input sequence into a single, fixed-size **Context Vector** before the Decoder begins generating.

![Information Bottleneck](outputs/information-bottleneck.png)

This architectural constraint creates an **Information Bottleneck**:
- A 64-dimensional vector can only store a finite amount of semantic representation.
- If the input is a 3-word sentence (e.g., *"Je suis fatigué"*), the bottleneck is wide enough.
- If the input is a 30-word sentence, compressing it into the same 64-dimensional vector results in severe information loss. The early words of the sentence "fade" out as the recurrent hidden state is repeatedly updated.

---

## 2. Concrete Mathematical Example

Let's illustrate why information decays recursively in recurrent encoders.

An Encoder updates its hidden state $h_t$ at each step using the weight matrix $U$ and the new word vector $x_t$:
$$h_t = \tanh(W x_t + U h_{t-1})$$

Let's trace a long sentence containing 12 words:
`"elle est heureuse de voir son ami petit à paris..."`
- Step 1: Input `"elle"` enters $\rightarrow$ updates state to $h_1$.
- Step 2: Input `"est"` enters $\rightarrow$ updates state to $h_2 = \tanh(W \cdot \text{"est"} + U h_1)$.
- Step 3: Input `"heureuse"` enters $\rightarrow$ updates state to $h_3 = \tanh(W \cdot \text{"heureuse"} + U h_2)$.
...
- Step 12: Input `"paris"` enters $\rightarrow$ updates state to $h_{12}$ (the final **Context Vector**).

Let's expand the recursive math back to $h_1$:
$$h_{12} \approx \tanh(W \cdot x_{12} + U \cdot \tanh(W \cdot x_{11} + U(\dots \tanh(W \cdot x_1 + \dots))))$$

Because of the repeated multiplication by the transition weight matrix $U$ (where values are typically $< 1$ or constrained by non-linear squashing activations like $\tanh$), the influence of the first word $x_1$ (`"elle"`) on the final state $h_{12}$ decays exponentially. 

By the time the Decoder receives $h_{12}$, the representation is heavily dominated by the latest processed words (like `"paris"`), and the historical trace of `"elle"` has faded. Consequently, the Decoder might fail to generate the correct subject pronoun (`"she"`), outputting `"he"` or getting lost.

---

## 3. The Solution: Transition to Attention

To resolve the information bottleneck, we need a mechanism that does not force the model to compress everything into one vector. 

Instead, the decoder should be allowed to **look back at all of the encoder's intermediate hidden states** at each step of the generation process, dynamically choosing which source words to focus on. 

This core concept — letting the model focus on relevant source states dynamically — is the **Attention Mechanism**, which we will implement and explore in the next phase.
